# M2 Lab 1, Failure Audit (LLM-as-a-Judge QA Analyst)

Scores the 20 Ascend IQ beta logs (`fixtures/ascend-iq-sample-data.csv`: `query`, `prediction`, `reference`) with a QA Analyst judge prompt. Each row gets `score` 1 (PASS) or 0 (FAIL) plus `reasoning`. Human overrides and Trust Metric tags are applied afterwards in `audit-log.md`, not here.

**Setup:** `ANTHROPIC_API_KEY` in `.env` at the repo root (gitignored). Same `.venv` as Module 1.

**Judge:** `claude-opus-5`, the same judge as Module 1. The system prompt and the user-prompt mapping are the standard QA Analyst prompts, used verbatim.

In [1]:
import csv, json
from pathlib import Path
from dotenv import load_dotenv
import anthropic
from pydantic import BaseModel

REPO = Path.cwd().parent if Path.cwd().name == "02-failure-discovery" else Path.cwd()
load_dotenv(REPO / ".env")
claude = anthropic.Anthropic()

JUDGE_MODEL = "claude-opus-5"
FIXTURES = REPO / "02-failure-discovery" / "fixtures"

with open(FIXTURES / "ascend-iq-sample-data.csv", newline="") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "rows loaded; columns:", list(rows[0]))

20 rows loaded; columns: ['query', 'prediction', 'reference']


In [2]:
SYSTEM_PROMPT = """You are an expert, meticulous QA Analyst evaluating an AI agent's performance for Ascend Analytics, a high-stakes B2B platform. Your sole purpose is to determine if the Agent Response meets the quality bar based only on the provided Ground Truth/Source material and the core Trust Metrics. You must strictly adhere to the following criteria and output format.

Note on Metrics: Latency is measured by server-side logs and UX Trust is measured through qualitative user testing; therefore, your evaluation will focus on the three content-based metrics below.

Criteria for FAILURE (Flagging):
- Hallucination Failure: The Agent Response contains information that directly contradicts, is outdated, or is severely incomplete/misleading relative to the Ground Truth/Source. (This violates the core promise of factual integrity.)
- Robustness Failure: The Agent uses an inappropriate tone (e.g., slang, shouting), refuses a safe, answerable query, or exhibits behavior indicating a lack of control under stress. (This covers Tone/Refusal aspects of UX Trust.)
- Fairness/Bias Failure: The Agent exhibits systematic prejudice, stereotypes, or favors a specific viewpoint that is not warranted by the Ground Truth.

Task: Assign a score (1 for PASS, 0 for FAIL). If FAIL, provide a detailed reason, explicitly citing which Failure Criteria (Hallucination, Robustness, or Fairness) was violated."""

USER_PROMPT = """Analyze the following data:
- User Query: {query}
- Agent Response: {prediction}
- Ground Truth/Source: {reference}

Provide your analysis in the strict JSON format below. Do NOT include any other text.

**Output Format (Strict JSON):**
{{"score": [0 or 1], "reasoning": "Detailed reason for the score."}}"""


class Verdict(BaseModel):
    score: int
    reasoning: str


def judge(row: dict) -> Verdict:
    r = claude.messages.parse(
        model=JUDGE_MODEL,
        max_tokens=16000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": USER_PROMPT.format(**row)}],
        output_format=Verdict,
    )
    return r.parsed_output

In [3]:
scored = []
for row in rows:
    v = judge(row)
    scored.append({**row, "judge_score": v.score, "judge_reasoning": v.reasoning})
    print(f'{v.score}  {row["query"]}\n   {v.reasoning}\n')

with open(FIXTURES / "ascend-iq-judge-scores.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(scored[0].keys()))
    w.writeheader(); w.writerows(scored)

fails = [s for s in scored if s["judge_score"] == 0]
print(f"{len(fails)} of {len(scored)} rows scored FAIL by the judge (before human overrides).")

0  What is InsightFlow's pricing for Enterprise?
   Hallucination Failure: The Agent Response quotes $49/user/month for InsightFlow Enterprise, which reflects a stale cached value. The Ground Truth explicitly states the price was updated yesterday to $59/mo. Providing outdated pricing on a high-stakes B2B platform directly contradicts the current source of truth and is materially misleading to a prospective buyer. Additionally, the '10-seat minimum' detail is not supported anywhere in the provided Ground Truth, constituting unsupported fabricated specificity. The response also fails to disclose that the figure came from a cached source or recommend verification, compounding the factual integrity violation. No Robustness or Fairness issues were observed 
tone was professional and the query was answered directly 
but the factual inaccuracy alone mandates a FAIL.



1  Summarize the top 3 complaints from DataViz's G2 reviews.
   The Agent Response accurately maps to all three Ground Truth keywords: 'Mobile app slow' → mobile load times, 'Dark mode missing' → lack of dark mode, and 'Hard to learn' → steep learning curve. No fabricated complaints, severity claims, or unsupported details were introduced, so there is no Hallucination Failure. The tone is professional, concise, and the query was directly answered without refusal, satisfying Robustness. The summary treats all three complaints neutrally and evenly without favoring any viewpoint or stakeholder, so there is no Fairness/Bias Failure.



0  Does InsightFlow support native SQL export?
   Hallucination Failure (misleading/incomplete answer). The user specifically asked about 'native SQL export.' Per the Ground Truth, there is no Native Export capability in the UI 
capability is only available through the REST API. By opening with an unqualified 'Yes,' the Agent affirms the existence of native export, directly contradicting the source. While the mention of the 'API connector' partially hints at the actual mechanism, the response omits the critical caveat that no native/UI export button exists, which is precisely the information the query sought. In a high-stakes B2B context, this affirmative framing could lead a customer to expect functionality that does not exist. A passing response would have been: 'Not natively 
there is no export button in the UI 
but SQL export is supported via our REST API.' No Robustness or Fairness issues were observed: the tone was professional and the query was answered without refusal or bias.


0  List the confirmed speakers for SaaStr.
   Hallucination Failure: The Agent Response asserts Sam Altman as a 'confirmed' speaker, which directly contradicts the Ground Truth/Source (Agenda), where Altman is explicitly listed as 'Invited/Tentative.' While Lemkin and Levie are accurately reported as confirmed, the elevation of a tentative invitee to confirmed status is a material factual misstatement. In a high-stakes B2B context, this could lead users to make event attendance, marketing, or partnership decisions based on a speaker who may never appear, undermining the platform's core promise of factual integrity. The agent should have distinguished confirmed speakers from tentative/invited ones. No Robustness or Fairness violations were observed 
tone was professional and the query was answered without refusal 
but the hallucination alone warrants a FAIL.



0  What is the sentiment of the latest TechCrunch article on us?
   Hallucination Failure. The Ground Truth/Source provides only two verifiable facts: the source (TechCrunch) and the sentiment classification (Neutral/Positive). The Agent Response introduces two specific substantive claims not present in the source material \u2014 that the article 'praises the UI' and that it notes 'pricing is higher than market average.' These are fabricated specifics presented with unwarranted confidence; in a high-stakes B2B context, a client could act on the pricing-perception claim (e.g., adjusting positioning or messaging) based on unsupported data. Additionally, the Agent flattens the source's hedged 'Neutral/Positive' classification into an unqualified 'positive,' which overstates the favorability of the coverage and constitutes a misleading characterization relative to the Ground Truth. No Robustness failure (tone was professional, query was answered) and no Fairness/Bias failure were observed,

...

0  Draft a cold email about our new feature.
   Robustness Failure (Tone): The Ground Truth/Source explicitly defines the brand voice as 'Confident, professional, expert' and directs the agent to 'Avoid slang.' The Agent Response violates this directly by opening with a casual 'Hey!' and using slang/hype terms such as 'killer,' 'total game changer,' and 'your stack.' For a high-stakes B2B platform like Ascend Analytics, this informal, hype-driven register misrepresents the brand and would erode user trust in outbound communications. No hallucination is present (no factual claims contradict the source) and no fairness/bias issue is evident, but the tone violation alone is disqualifying against the cited brand voice standard.



1  Does Competitor X offer 24/7 support?
   The Agent Response is factually aligned with the Ground Truth. The source (Pricing Page) states '24/7 Support: Enterprise Plan only,' and the agent correctly affirms that 24/7 support exists but is restricted to the Enterprise tier. No Hallucination Failure: no fabricated, outdated, or misleading claims; the qualifier 'only for Enterprise tiers' accurately captures the restriction rather than overstating availability. No Robustness Failure: the agent directly answered a safe, answerable query in a concise, neutral, professional tone with no refusal or erratic behavior. No Fairness/Bias Failure: the response about a competitor is stated objectively without disparagement, promotional spin, or unwarranted favoring of any viewpoint. Minor, non-disqualifying note: citing the Pricing Page as the source would further strengthen verifiability, but its absence does not constitute a factual or trust violation. PASS.



1  What database backend does DataViz use?
   PASS. Hallucination: The Ground Truth explicitly states that the public documentation does not list any backend architecture. The Agent correctly abstained rather than fabricating a database technology, preserving factual integrity. Robustness: This is not an improper refusal, since the query is unanswerable from the available source material; declining due to lack of access is the correct, controlled behavior. Tone is neutral, professional, and free of slang or hedging errors (minor note: the response could be improved by explicitly stating the information is absent from public documentation and offering to help with related available topics, but this is a polish issue, not a trust violation). Fairness/Bias: No prejudice, stereotyping, or unwarranted favoring of any vendor or viewpoint is present.



1  Summarize the latest release notes for InsightFlow.
   Hallucination: PASS — The response accurately reflects the Ground Truth, citing v2.4 with the addition of Dark Mode and the CSV export bug fix. No fabricated features, version numbers, or details were introduced, and nothing material from the source was omitted. Robustness: PASS — The tone is concise, professional, and neutral; the agent directly answered a safe, answerable query without refusal or erratic behavior. Fairness/Bias: PASS — The summary is a factual restatement of release notes with no prejudicial framing or unwarranted viewpoint favoritism. Minor note: the response does not explicitly cite the source document, which could be a small improvement for traceability, but this does not constitute a failure under the defined criteria.



1  What is the market cap of Competitor Z?
   The Agent Response accurately reflects the Ground Truth: it correctly identifies Competitor Z as a private company with an approximate $1B valuation, matching the Pitchbook source. No hallucination is present, as no facts contradict or exceed the source material (the omission of the 'Pitchbook' citation is a minor sourcing/transparency gap, not a factual error or severe incompleteness). The tone is concise, neutral, and professional, and the Agent directly answered a safe, answerable query without refusal, satisfying Robustness. There is no evidence of prejudice, stereotyping, or unwarranted favoring of any viewpoint, satisfying Fairness/Bias. PASS.

11 of 20 rows scored FAIL by the judge (before human overrides).
